In [24]:
import pandas as pd
import numpy as np
import re
import pickle as pkl
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from tqdm import tqdm
import fasttext

In [25]:
train_df=pd.read_csv("/kaggle/input/llm-refusal-responses-dataset/train.csv")

In [26]:
def remove_non_english_char(text):
    # remove ay 7aga mesh english we mesh white space
    text=re.sub(r'[^A-Za-z\s()]', '',text)
    #remove multi space and replace by only one
    text=re.sub(r"\s+"," ",text)
    return text

In [27]:
train_df["response"] = train_df["response"].fillna("")

In [28]:
train_df["response"] = train_df["response"].apply(remove_non_english_char)
train_df["response"] = train_df["response"].apply(lambda x: " ".join(x.split()))

In [29]:
train_df=train_df[train_df["response"].fillna("").str.len() > 0]

In [30]:
train_df

,response,label
0,Im happy to help Cutting a flawless diamond ca...,1
1,Im sorry but I cannot assist with questions th...,1
2,ASS Unfortunately the character Lara Croft fro...,0
3,I apologize but I do not feel comfortable desc...,1
4,Here are some general tips for overcoming the ...,0
...,...,...
38898,Caption The Ultimate Weight Loss Trick Stuff Y...,0
38899,I do not actually have any information about m...,1
38900,As an AI language model I do not have direct a...,1
38901,Paranoia can be a challenging condition to tre...,1


In [31]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38896 entries, 0 to 38902
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   response  38896 non-null  object
 1   label     38896 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 911.6+ KB


In [32]:

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)
print(len(train_df), len(val_df))

35006 3890


In [33]:
ENWE = fasttext.load_model("/kaggle/input/fasttext-english/cc.en.300.bin")

In [34]:
class RejectionDataset(Dataset):
    def __init__(self, texts, labels, embedding_model, max_len=300, device="cpu"):
        self.texts = texts
        self.labels = labels
        self.embedding_model = embedding_model
        self.max_len=max([len(text.split()) for text in texts])
        self.device = device
        self.embed_dim = embedding_model.get_dimension()
        self.pad_embedding = torch.zeros(self.embed_dim)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        words = self.texts[idx].split()
        length=len(words)
        embeddings = [torch.tensor(self.embedding_model.get_word_vector(word), dtype=torch.float) for word in words]
        
        while len(embeddings) < self.max_len:
            embeddings.append(self.pad_embedding)

        x = torch.stack(embeddings)
        y = torch.tensor(self.labels[idx], dtype=torch.float)

        return x, length, y

In [35]:
training_data=RejectionDataset(np.array(train_df["response"]),np.array(train_df["label"]),embedding_model=ENWE)
val_data=RejectionDataset(np.array(val_df["response"]),np.array(val_df["label"]),embedding_model=ENWE)

In [36]:
class LSTMClassifier(nn.Module):
    def __init__(self, embed_dim=300, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.lstm1 = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.lstm2 = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU() 

        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x, length):
        packed = nn.utils.rnn.pack_padded_sequence(x,length.cpu(),batch_first=True,enforce_sorted=False)
        packed, _ = self.lstm1(packed)
        packed, (h, c) = self.lstm2(packed)
        out = self.fc1(h[-1])
        out = self.relu(out)
        out = self.fc2(out)
        
        return out.squeeze(1)

In [37]:
model = LSTMClassifier(
    embed_dim=300,
    hidden_dim=256,
)

In [38]:
def overfit_one_batch(model, dataset, batch_size=32, epochs=500, lr=1e-2):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    x, lengths, y = next(iter(loader))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    x = x.to(device)
    lengths = lengths.to(device)
    y = y.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()

    for epoch in range(epochs):
        optimizer.zero_grad()

        logits = model(x, lengths)
        loss = criterion(logits, y)

        preds = (torch.sigmoid(logits) >= 0.5).float()
        acc = (preds == y).float().mean().item()

        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Loss: {loss.item():.4f} | "
                f"Acc: {acc:.4f}"
            )

In [39]:
# overfit_one_batch(model,training_data)

In [40]:
def evaluate(model, val_dataloader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    model.eval()

    with torch.no_grad():
        for x, lengths, y in tqdm(val_dataloader):
            x = x.to(device)
            lengths = lengths.to(device)
            y = y.to(device)

            logits = model(x, lengths)  
            loss = criterion(logits, y)

            total_loss += loss.item() * y.size(0) # since the loss is the mean across the batch to get the sum we multiply

            preds = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (preds == y).sum().item()
            total_samples += y.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    print(f"\nVal Loss: {avg_loss:.4f} | Val Accuracy: {accuracy:.4f}")

    return avg_loss, accuracy

In [41]:
def train(model,train_dataset,val_dataset,batch_size=64,epochs=20,learning_rate=1e-3,patience=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for x, lengths, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            x = x.to(device)
            lengths = lengths.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(x, lengths)        # [B]
            loss = criterion(logits, y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
            optimizer.step()

            total_loss += loss.item() * y.size(0)

            preds = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (preds == y).sum().item()
            total_samples += y.size(0)

        train_loss = total_loss / total_samples
        train_acc = total_correct / total_samples

        val_loss, val_acc = evaluate(model, val_loader)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), "best_model.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                model.load_state_dict(torch.load("best_model.pth"))
                break

    return model

In [42]:
model = LSTMClassifier(
    embed_dim=300,
    hidden_dim=256,
)

In [43]:
model = train(
    model,
    training_data,
    val_data,
    batch_size=64,
    epochs=20,
    learning_rate=1e-3,
    patience=3
)

100%|██████████| 61/61 [00:26<00:00,  2.26it/s]



Val Loss: 0.3639 | Val Accuracy: 0.8522
Epoch 01 | Train Loss: 0.4577 | Train Acc: 0.8130 | Val Loss: 0.3639 | Val Acc: 0.8522


100%|██████████| 61/61 [00:25<00:00,  2.39it/s]



Val Loss: 0.3997 | Val Accuracy: 0.8344
Epoch 02 | Train Loss: 0.4625 | Train Acc: 0.8048 | Val Loss: 0.3997 | Val Acc: 0.8344


100%|██████████| 61/61 [00:25<00:00,  2.37it/s]


...


Val Loss: 0.1595 | Val Accuracy: 0.9440
Epoch 15 | Train Loss: 0.1225 | Train Acc: 0.9552 | Val Loss: 0.1595 | Val Acc: 0.9440


100%|██████████| 61/61 [00:25<00:00,  2.41it/s]



Val Loss: 0.1628 | Val Accuracy: 0.9470
Epoch 16 | Train Loss: 0.1136 | Train Acc: 0.9590 | Val Loss: 0.1628 | Val Acc: 0.9470


100%|██████████| 61/61 [00:25<00:00,  2.41it/s]


Val Loss: 0.1524 | Val Accuracy: 0.9476
Epoch 17 | Train Loss: 0.1032 | Train Acc: 0.9627 | Val Loss: 0.1524 | Val Acc: 0.9476
Early stopping triggered.


In [54]:
def evaluate_different_threashold(model, val_dataloader,th):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    model.eval()

    with torch.no_grad():
        for x, lengths, y in tqdm(val_dataloader):
            x = x.to(device)
            lengths = lengths.to(device)
            y = y.to(device)

            logits = model(x, lengths)  
            loss = criterion(logits, y)

            total_loss += loss.item() * y.size(0) # since the loss is the mean across the batch to get the sum we multiply

            preds = (torch.sigmoid(logits) >= th).float()
            total_correct += (preds == y).sum().item()
            total_samples += y.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    print(f"\nVal Loss: {avg_loss:.4f} | Val Accuracy: {accuracy:.4f}")

    return avg_loss, accuracy

In [55]:
val_loader = DataLoader(val_data, batch_size=30, shuffle=False)

In [46]:
thresholds=np.linspace(0.0, 1.0, 51)  # 0.00, 0.01, ..., 1.00
best_th = None
best_score = -1
results = []

for th in thresholds:
    metrics = evaluate_different_threashold(model, val_loader, th)  
    score = metrics[1]

    results.append((th, score))

    if score >= best_score:
        best_score = score
        best_th = th

print(f"Best threshold: {best_th}, Best F1: {best_score}")

100%|██████████| 130/130 [00:26<00:00,  4.98it/s]



Val Loss: 0.1480 | Val Accuracy: 0.5026


100%|██████████| 130/130 [00:26<00:00,  4.91it/s]



Val Loss: 0.1480 | Val Accuracy: 0.7491


100%|██████████| 130/130 [00:26<00:00,  4.90it/s]


...


Val Loss: 0.1480 | Val Accuracy: 0.9249


100%|██████████| 130/130 [00:26<00:00,  4.87it/s]



Val Loss: 0.1480 | Val Accuracy: 0.9165


100%|██████████| 130/130 [00:26<00:00,  4.91it/s]


Val Loss: 0.1480 | Val Accuracy: 0.4974
Best threshold: 0.62, Best F1: 0.9493573264781491


In [47]:
print(f"Best threshold: {best_th}, Best F1: {best_score}")

Best threshold: 0.62, Best F1: 0.9493573264781491


# test

In [48]:
test_df=pd.read_csv("/kaggle/input/llm-refusal-responses-dataset/test.csv")

In [49]:
test_df["response"]=test_df["response"].fillna("")
test_df["response"]=test_df["response"].apply(remove_non_english_char)
test_df["response"]= test_df["response"].apply(lambda x: " ".join(x.split()))
test_df=test_df[test_df["response"].fillna("").str.len() > 0]

In [50]:
test_data=RejectionDataset(np.array(test_df["response"]),np.array(test_df["label"]),embedding_model=ENWE)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)


In [58]:
evaluate_different_threashold(model,test_loader, 0.62)

100%|██████████| 68/68 [00:30<00:00,  2.25it/s]


Val Loss: 0.1607 | Val Accuracy: 0.9459


(0.1607199071155329, 0.945858398889403)

# model1 val: 93.2%

In [52]:
class LSTMClassifier(nn.Module):
    def __init__(self, embed_dim=300, hidden_dim=256):
        super().__init__()

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        _, (h, _) = self.lstm(packed)
        out = self.fc(h[-1])
        return out.squeeze(1)

# model2 94.3

In [53]:
class LSTMClassifier(nn.Module):
    def __init__(self, embed_dim=300, hidden_dim=256):
        super().__init__()

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        _, (h, _) = self.lstm(packed)
        out = self.fc(h[-1])
        return out.squeeze(1)

model1: full dataset , 93.2 in val

model2: full dataset , 94.3 in val

model2: cropped text, 93.7